In [3]:
# =========================
# 1. IMPORTS
# =========================
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import ipywidgets as widgets
from IPython.display import display

# =========================
# 2. CARGAR DATASET
# =========================
df = pd.read_csv('spam.csv', encoding='latin-1')

df.columns = ['label', 'message', 'x1', 'x2', 'x3']
df = df[['label', 'message']]

df['label'] = df['label'].str.strip().str.lower()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

df = df.dropna()

# =========================
# 3. AGREGAR ESPAÑOL (CLAVE)
# =========================
nuevos_datos = pd.DataFrame({
    'label': [1]*15 + [0]*15,
    'message': [
        # 🔴 SPAM (español)
        "Gana dinero rapido ahora",
        "Has ganado un premio reclama ya",
        "Felicidades eres ganador de un premio",
        "Oferta exclusiva haz clic ahora",
        "Llama ahora para reclamar tu premio",
        "Dinero gratis disponible ahora",
        "Haz clic para ganar premios",
        "Promocion limitada gana dinero",
        "Has sido seleccionado para recompensa",
        "Reclama tu bono ahora mismo",
        "Premio garantizado llama ya",
        "Oferta limitada gana ahora",
        "Dinero facil disponible",
        "Accede ahora a tu premio",
        "Ultima oportunidad gana dinero",

        # 🟢 NO SPAM (español)
        "Hola como estas",
        "Nos vemos mañana",
        "Te envio el informe",
        "Hablamos luego",
        "Estoy en casa",
        "Vamos a estudiar",
        "Nos vemos en clase",
        "Gracias por la ayuda",
        "Te llamo mas tarde",
        "Que haces hoy",
        "Voy en camino",
        "Ya llegue a casa",
        "Te escribo luego",
        "Todo bien por aqui",
        "Nos vemos pronto"
    ]
})

df = pd.concat([df, nuevos_datos], ignore_index=True)

# =========================
# 4. DIVISIÓN
# =========================
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 5. VECTORIZACIÓN (SIN STOPWORDS)
# =========================
vectorizador = TfidfVectorizer(
    ngram_range=(1,2)
)

X_train_vec = vectorizador.fit_transform(X_train)
X_test_vec = vectorizador.transform(X_test)

# =========================
# 6. MODELO
# =========================
modelo = MultinomialNB()
modelo.fit(X_train_vec, y_train)

print("✅ Modelo entrenado")

# =========================
# 7. FUNCIÓN DE PRUEBA
# =========================
def probar(texto):
    x = vectorizador.transform([texto])
    prob = modelo.predict_proba(x)[0]
    spam_prob = prob[1]

    print(f"\nMensaje: {texto}")
    print(f"Prob SPAM: {spam_prob:.2f}")

    if spam_prob > 0.4:
        print("👉 📧 SPAM")
    else:
        print("👉 ✅ NO SPAM")

# =========================
# 8. PRUEBAS
# =========================
probar("Free entry!!! win now")
probar("URGENT! call now")
probar("Hello, how are you?")
probar("See you tomorrow")

# Español
probar("Gana dinero ahora")
probar("Has ganado un premio")
probar("Hola como estas")
probar("Nos vemos mañana")

# =========================
# 9. GUARDAR MODELO
# =========================
with open('modelo.pkl', 'wb') as f:
    pickle.dump(modelo, f)

with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizador, f)

print("\n✅ Modelo guardado")

# =========================
# 10. INTERFAZ
# =========================
def predecir(texto):
    datos = vectorizador.transform([texto])
    prob = modelo.predict_proba(datos)[0]
    spam_prob = prob[1]

    if spam_prob > 0.4:
        return f"📧 Spam ({spam_prob:.2f})"
    else:
        return f"✅ No Spam ({1-spam_prob:.2f})"

entrada = widgets.Textarea(
    placeholder='Escribe el mensaje...',
    description='Mensaje:',
    layout=widgets.Layout(width='80%', height='100px')
)

boton = widgets.Button(description='Predecir', button_style='success')
salida = widgets.Output()

def click(b):
    with salida:
        salida.clear_output()
        print(predecir(entrada.value))

boton.on_click(click)

display(entrada, boton, salida)

✅ Modelo entrenado

Mensaje: Free entry!!! win now
Prob SPAM: 0.66
👉 📧 SPAM

Mensaje: URGENT! call now
Prob SPAM: 0.41
👉 📧 SPAM

Mensaje: Hello, how are you?
Prob SPAM: 0.00
👉 ✅ NO SPAM

Mensaje: See you tomorrow
Prob SPAM: 0.01
👉 ✅ NO SPAM

Mensaje: Gana dinero ahora
Prob SPAM: 0.60
👉 📧 SPAM

Mensaje: Has ganado un premio
Prob SPAM: 0.45
👉 📧 SPAM

Mensaje: Hola como estas
Prob SPAM: 0.13
👉 ✅ NO SPAM

Mensaje: Nos vemos mañana
Prob SPAM: 0.06
👉 ✅ NO SPAM

✅ Modelo guardado


Textarea(value='', description='Mensaje:', layout=Layout(height='100px', width='80%'), placeholder='Escribe el…

Button(button_style='success', description='Predecir', style=ButtonStyle())

Output()